In [1]:
import os
import sys
import json
import time
import signal
import subprocess
from pathlib import Path

REPO = "https://github.com/eslam-ahmed43/llm-lab.git"
ROOT = Path("/kaggle/working/llm-lab")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
(ROOT / "logs").mkdir(exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
for name in ["exp7_launch_model.py", "exp7_fit.py"]:
    assert (ROOT / "src" / name).exists(), f"{name} is missing: push it to GitHub first"

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log --oneline -1; ls src", shell=True, capture_output=True, text=True).stdout)
print(sys.version)

Cloning into '/kaggle/working/llm-lab'...


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

7e4998b Add Exp 7 scripts: launch-aware decode model
bench_inference.py
exp5_internals.py
exp6_profiling.py
exp7_fit.py
exp7_launch_model.py
hf_batched_server.py
hf_server.py
plot_exp6.py
plot_lora.py
quantize.py
train_lora.py

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
%%bash
set -e
python -c "
from huggingface_hub import snapshot_download
for m in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct']:
    snapshot_download(m)
    print('cached', m)
"

cached Qwen/Qwen2.5-0.5B-Instruct
cached Qwen/Qwen2.5-1.5B-Instruct
cached Qwen/Qwen2.5-3B-Instruct


Fetching 12 files: 100%|██████████| 12/12 [00:37<00:00,  3.13s/it]


In [3]:
def run_script(script, tag, args, timeout_s=1800):
    log_path = ROOT / "logs" / f"exp7_{tag}.log"
    cmd = [sys.executable, "-X", "faulthandler", "-u", str(ROOT / "src" / script)] + args
    env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}

    def new_lines(shown):
        lines = log_path.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.strip() and "Loading weights" not in line and "unauthenticated" not in line:
                print(line[:400], flush=True)
        return len(lines)

    start, shown = time.time(), 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(ROOT))
        while proc.poll() is None:
            time.sleep(5)
            shown = new_lines(shown)
            if time.time() - start > timeout_s:
                os.kill(proc.pid, signal.SIGABRT)
                time.sleep(3)
                proc.kill()
                print(log_path.read_text(errors="replace")[-3000:])
                raise TimeoutError(f"{tag} exceeded {timeout_s}s")
    new_lines(shown)
    if proc.returncode != 0:
        print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{tag} failed with exit code {proc.returncode}")

In [4]:
run_script("exp7_launch_model.py", "smoke",
           ["--model", "Qwen/Qwen2.5-0.5B-Instruct", "--check", "--repeats", "2",
            "--out", str(ROOT / "results/launch_smoke.json")])

[check] fp32 equivalence of the decode variants
   {'dynamic_nosync': True, 'static_eager': True, 'static_graph': True}
[fp16] building Qwen/Qwen2.5-0.5B-Instruct
  0.49B parameters, 24 layers, weights 0.99 GB, bandwidth 241 GB/s, weight-streaming floor 4.1 ms
  dynamic_eager_nosync: 25.82 ms/step (enqueue alone 25.82 ms)
  static_eager_nosync: 26.05 ms/step (enqueue alone 26.04 ms)
  static_cuda_graph: 9.48 ms/step (enqueue alone 8.98 ms)
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  profile dynamic_eager_nosync: 1128 device operations per step, GPU busy 8.04 ms
  profile static_eager_nosync: 1202 device operations per step, GPU busy 15.52 ms
  profile static_cuda_graph: 1204 device operations per step, GPU busy 12.42 ms
saved /kaggle/working/llm-lab/results/launch_smoke.js

In [5]:
import traceback

(ROOT / "results/launch_smoke.json").unlink(missing_ok=True)
runs = [("0p5b", "Qwen/Qwen2.5-0.5B-Instruct", ["--check"]),
        ("1p5b", "Qwen/Qwen2.5-1.5B-Instruct", []),
        ("3b", "Qwen/Qwen2.5-3B-Instruct", [])]
for tag, model, extra in runs:
    out = ROOT / "results" / f"launch_{tag}.json"
    if out.exists():
        print(tag, "already done")
        continue
    try:
        run_script("exp7_launch_model.py", tag, ["--model", model, "--out", str(out)] + extra, timeout_s=2400)
    except Exception:
        traceback.print_exc()

[check] fp32 equivalence of the decode variants
   {'dynamic_nosync': True, 'static_eager': True, 'static_graph': True}
[fp16] building Qwen/Qwen2.5-0.5B-Instruct
  0.49B parameters, 24 layers, weights 0.99 GB, bandwidth 242 GB/s, weight-streaming floor 4.1 ms
  dynamic_eager_nosync: 24.73 ms/step (enqueue alone 24.72 ms)
  static_eager_nosync: 25.43 ms/step (enqueue alone 25.42 ms)
  static_cuda_graph: 9.72 ms/step (enqueue alone 9.21 ms)
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  profile dynamic_eager_nosync: 1128 device operations per step, GPU busy 8.04 ms
  profile static_eager_nosync: 1202 device operations per step, GPU busy 13.72 ms
  profile static_cuda_graph: 1204 device operations per step, GPU busy 11.80 ms
saved /kaggle/working/llm-lab/results/launch_0p5b.jso

In [6]:
run_script("exp7_fit.py", "fit", ["--results-dir", str(ROOT / "results"), "--reference", "1.5B",
                                   "--out", str(ROOT / "results/launch_model_fit.json"),
                                   "--plot", str(ROOT / "plots/launch_model.png")])

fitted on Qwen/Qwen2.5-1.5B-Instruct: c_layer 272 us, 49.9 operations per layer, t_launch 21.6 us per operation
model                        meas eager    pred   err%   meas graph    pred   err%   speedup meas/pred  regime meas/pred
Qwen/Qwen2.5-0.5B-Instruct        25.43   25.88   +1.8         9.72   10.64   +9.4      2.61 /  2.43  CPU-bound / CPU-bound
Qwen/Qwen2.5-1.5B-Instruct        30.20   30.20   +0.0        20.43   20.43   +0.0      1.48 /  1.48  CPU-bound / CPU-bound  (fit)
Qwen/Qwen2.5-3B-Instruct          38.71   38.83   +0.3        35.41   35.41   -0.0      1.09 /  1.10  CPU-bound / CPU-bound
baselines, error of the eager step (%): Qwen2.5-0.5B-Instruct bandwidth-only -62, layers-only +2; Qwen2.5-3B-Instruct bandwidth-only +56, layers-only +0
saved /kaggle/working/llm-lab/results/launch_model_fit.json and /kaggle/working/llm-lab/plots/launch_model.png


In [7]:
import zipfile

with zipfile.ZipFile("/kaggle/working/llm-lab-exp7.zip", "w", zipfile.ZIP_DEFLATED) as z:
    files = sorted((ROOT / "results").glob("launch_*.json"))
    files += sorted((ROOT / "plots").glob("launch_model.png")) + sorted((ROOT / "logs").glob("exp7_*.log"))
    for f in files:
        z.write(f, arcname=f"{f.parent.name}/{f.name}")
print("Zipped:", zipfile.ZipFile("/kaggle/working/llm-lab-exp7.zip").namelist())

Zipped: ['results/launch_0p5b.json', 'results/launch_1p5b.json', 'results/launch_3b.json', 'results/launch_model_fit.json', 'plots/launch_model.png', 'logs/exp7_0p5b.log', 'logs/exp7_1p5b.log', 'logs/exp7_3b.log', 'logs/exp7_fit.log', 'logs/exp7_smoke.log']
